# Proyecto Minería 4.0 – Fase 3

## Machine Learning espacial para leyes de Au – ALPACAY

Notebook reproducible para **Jupyter, Google Colab y un entorno Python de Antigravity**.

### Objetivo

Comparar modelos de regresión para estimar `Au_g_t` utilizando información espacial, potencia y contexto operativo. La evaluación principal separa bloques espaciales completos para reducir la fuga de información entre muestras vecinas.

### Principios

- Una sola zona analítica: **ALPACAY**.
- `AVISPA` se conserva únicamente como etiqueta histórica.
- `GA9920` permanece como tramo alejado de ALPACAY.
- Solo ingresan registros con coordenadas completas, Au numérico y `IncluirML = SI`.
- BLANCO y DUPLICADO permanecen fuera del entrenamiento.
- Las leyes altas no se eliminan automáticamente; el objetivo se transforma con `log1p`.
- La validación espacial es la referencia. La validación aleatoria se muestra para evidenciar optimismo potencial.

> Este modelo es experimental. No constituye estimación de recursos o reservas ni reemplaza la interpretación geológica.


In [ ]:
from pathlib import Path
import base64
import json
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

try:
    from IPython.display import display, HTML
except ImportError:
    def display(obj): print(obj)
    class HTML(str): pass

try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    files = None
    EN_COLAB = False

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")
SEMILLA = 42
NAVY, AZUL, ORO, VERDE, ROJO = "#17365D", "#4472C4", "#D6A100", "#008C7A", "#C00000"
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter/Antigravity/local")


## 1. Carga del Excel

Usa preferentemente `FASE2_MODELO_POWERBI_ALPACAY_CORREGIDO_INTERNO.xlsx`. También se admite el libro QA/QC de Fase 1.

En Colab se abrirá el selector de archivos. En un entorno local, coloca el Excel junto al notebook o en una subcarpeta `data`.


In [ ]:
def seleccionar_excel():
    if EN_COLAB:
        cargados = files.upload()
        excels = [Path(nombre) for nombre in cargados if nombre.lower().endswith((".xlsx", ".xls"))]
        if not excels:
            raise FileNotFoundError("No se cargó ningún archivo Excel.")
        return excels[0]

    grupos = [Path.cwd().glob("*.xlsx"), (Path.cwd() / "data").glob("*.xlsx")]
    candidatos = [p for grupo in grupos for p in grupo if not p.name.startswith("~$")]
    if not candidatos:
        raise FileNotFoundError("Coloca el Excel junto al notebook o dentro de data.")
    preferidos = [p for p in candidatos if "ALPACAY_CORREGIDO" in p.name.upper()]
    return preferidos[0] if preferidos else candidatos[0]

ARCHIVO_EXCEL = seleccionar_excel()
print("Archivo seleccionado:", ARCHIVO_EXCEL.name)


## 2. Preparación de datos

La fuente se mantiene intacta. La zona se normaliza a ALPACAY y las coordenadas se convierten a valores relativos antes del modelamiento.


In [ ]:
def limpiar_texto(serie):
    return serie.astype("string").str.strip().replace({"": pd.NA, "NAN": pd.NA, "<NA>": pd.NA})


def cargar_datos(ruta):
    libro = pd.ExcelFile(ruta)
    if "Fact_Muestras" in libro.sheet_names:
        df = pd.read_excel(ruta, sheet_name="Fact_Muestras")
        for hoja, llave in [("Dim_Zona", "ZonaKey"), ("Dim_Tipo", "TipoKey"), ("Dim_Labor", "LaborKey"), ("Dim_Grupo", "GrupoKey")]:
            if hoja in libro.sheet_names and llave in df.columns:
                dim = pd.read_excel(ruta, sheet_name=hoja)
                columnas = [c for c in dim.columns if c == llave or c not in df.columns]
                df = df.merge(dim[columnas], on=llave, how="left")
        if "FechaKey" in df.columns:
            df["Fecha"] = pd.to_datetime(df["FechaKey"].astype("Int64").astype("string"), format="%Y%m%d", errors="coerce")
        fuente = "Modelo Power BI"
    else:
        hoja = "DATA_LIMPIA" if "DATA_LIMPIA" in libro.sheet_names else "DATA_ORIGINAL"
        df = pd.read_excel(ruta, sheet_name=hoja)
        df.columns = [str(c).strip() for c in df.columns]
        df = df.rename(columns={
            "FECHA": "Fecha", "CODIGO": "Codigo", "ZONA": "Zona", "VETA": "Veta",
            "TIPO": "Tipo", "LABOR": "Labor", "GRUPO": "Grupo", "Nv.": "Nivel",
            "POT (m)": "Potencia_m", "Au (gr/TM)": "Au_Reporte",
            "ESTE": "Este", "NORTE": "Norte", "COTA": "Cota", "AZ": "Azimut",
            "FLAG_COORD": "FlagCoord", "FLAG_AU": "FlagAu", "FLAG_TIPO": "FlagTipo",
            "INCLUIR_ML": "IncluirML", "MOTIVO_EXCLUSION": "MotivoExclusion",
        })
        df["Fecha"] = pd.to_datetime(df.get("Fecha"), errors="coerce")
        fuente = f"Fase 1 – {hoja}"

    if "MuestraKey" not in df.columns:
        df["MuestraKey"] = np.arange(1, len(df) + 1)
    for columna in ["Zona", "Veta", "Tipo", "Labor", "Grupo"]:
        if columna not in df.columns: df[columna] = pd.NA
        df[columna] = limpiar_texto(df[columna])
    for columna in ["Este", "Norte", "Cota", "Nivel", "Potencia_m", "Azimut"]:
        if columna not in df.columns: df[columna] = np.nan
        df[columna] = pd.to_numeric(df[columna], errors="coerce")
    if "Au_g_t" not in df.columns:
        df["Au_g_t"] = pd.to_numeric(df.get("Au_Reporte"), errors="coerce")
    else:
        df["Au_g_t"] = pd.to_numeric(df["Au_g_t"], errors="coerce")

    coord_ok = df[["Este", "Norte", "Cota"]].notna().all(axis=1)
    tipo_norm = df["Tipo"].fillna("").str.upper()
    es_qaqc = tipo_norm.str.contains("BLANCO|DUPLICADO", regex=True)
    if "IncluirML" not in df.columns:
        df["IncluirML"] = np.where(coord_ok & df["Au_g_t"].notna() & ~es_qaqc, "SI", "NO")

    df["Zona_original"] = df["ZonaOriginal"] if "ZonaOriginal" in df.columns else df["Zona"]
    df["Zona_analitica"] = "ALPACAY"
    return df, fuente


datos, FUENTE = cargar_datos(ARCHIVO_EXCEL)
candidatos = datos.loc[
    datos["IncluirML"].astype("string").str.upper().eq("SI")
    & datos["Au_g_t"].notna()
    & datos[["Este", "Norte", "Cota"]].notna().all(axis=1)
].copy().reset_index(drop=True)

if len(candidatos) < 100:
    raise ValueError(f"Solo hay {len(candidatos)} candidatos; revisar criterios antes de entrenar.")

origen = {c: float(candidatos[c].min()) for c in ["Este", "Norte", "Cota"]}
candidatos["Este_rel_m"] = candidatos["Este"] - origen["Este"]
candidatos["Norte_rel_m"] = candidatos["Norte"] - origen["Norte"]
candidatos["Cota_rel_m"] = candidatos["Cota"] - origen["Cota"]

xy = candidatos[["Este", "Norte"]].to_numpy()
pca = PCA(n_components=2).fit(xy)
candidatos[["Longitudinal_m", "Transversal_m"]] = pca.transform(xy)

componentes = DBSCAN(eps=20.0, min_samples=1).fit_predict(xy)
principal = pd.Series(componentes).value_counts().idxmax()
candidatos["Tramo_espacial"] = np.where(componentes == principal, "Principal", "Alejado")

print(FUENTE)
print(f"Registros totales: {len(datos):,}")
print(f"Candidatos ML: {len(candidatos):,}")
display(candidatos[["MuestraKey", "Zona_original", "Zona_analitica", "Labor", "Nivel", "Au_g_t", "Tramo_espacial"]].head())


## 3. Bloques de validación espacial

Los puntos se ordenan a lo largo del eje principal de la veta y se dividen en cinco bloques contiguos. Todas las muestras de una misma coordenada XY permanecen en el mismo bloque.


In [ ]:
puntos_unicos = candidatos[["Este", "Norte", "Longitudinal_m"]].drop_duplicates(["Este", "Norte"]).copy()
puntos_unicos["Bloque_espacial"] = pd.qcut(
    puntos_unicos["Longitudinal_m"].rank(method="dense"),
    q=5, labels=False, duplicates="drop"
).astype(int)
candidatos = candidatos.merge(
    puntos_unicos[["Este", "Norte", "Bloque_espacial"]], on=["Este", "Norte"], how="left"
)

resumen_bloques = candidatos.groupby("Bloque_espacial").agg(
    Muestras=("MuestraKey", "size"),
    Au_mediana=("Au_g_t", "median"),
    Au_promedio=("Au_g_t", "mean"),
    Este_rel_min=("Este_rel_m", "min"),
    Este_rel_max=("Este_rel_m", "max"),
).reset_index()

display(resumen_bloques.round(3))
assert candidatos["Bloque_espacial"].nunique() >= 4


## 4. Variables y modelos

El objetivo se transforma como `log1p(Au)` para reducir la influencia extrema de leyes muy altas. Las predicciones se devuelven posteriormente a g/t.

Modelos evaluados:

1. Mediana global: referencia mínima obligatoria.
2. Ridge: relación lineal regularizada.
3. Random Forest: relaciones no lineales.
4. Gradient Boosting: ensamble no lineal adicional.


In [ ]:
variables_numericas = [
    "Este_rel_m", "Norte_rel_m", "Cota_rel_m", "Longitudinal_m", "Transversal_m",
    "Potencia_m", "Azimut",
]
variables_categoricas = ["Labor", "Nivel", "Grupo", "Tramo_espacial"]
variables_modelo = variables_numericas + variables_categoricas

X = candidatos[variables_modelo].copy()
for columna in variables_categoricas:
    X[columna] = X[columna].astype(object).where(X[columna].notna(), np.nan)
X["Nivel"] = X["Nivel"].map(lambda v: str(v) if pd.notna(v) else np.nan)
y = candidatos["Au_g_t"].to_numpy(dtype=float)
y_log = np.log1p(y)
grupos = candidatos["Bloque_espacial"].to_numpy()

prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), variables_numericas),
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), variables_categoricas),
])

modelos = {
    "Base mediana": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=350, min_samples_leaf=5, max_features=0.75,
        random_state=SEMILLA, n_jobs=-1,
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=12, l2_regularization=1.0, random_state=SEMILLA,
    ),
}

pipelines = {nombre: Pipeline([("preparar", prep), ("modelo", modelo)]) for nombre, modelo in modelos.items()}
print("Variables:", variables_modelo)


## 5. Comparación de validación aleatoria y espacial

La métrica principal para seleccionar el modelo es el **MAE espacial en g/t**. RMSE penaliza más los errores grandes y `R²` puede ser negativo cuando el modelo generaliza peor que una referencia constante.


In [ ]:
def evaluar_esquema(nombre_esquema, divisiones):
    filas = []
    predicciones = {}
    for nombre_modelo, pipeline in pipelines.items():
        oof = np.full(len(X), np.nan)
        for entrenamiento, prueba in divisiones:
            estimador = clone(pipeline)
            estimador.fit(X.iloc[entrenamiento], y_log[entrenamiento])
            pred_log = estimador.predict(X.iloc[prueba])
            oof[prueba] = np.maximum(0.0, np.expm1(pred_log))
        predicciones[nombre_modelo] = oof
        filas.append({
            "Validacion": nombre_esquema,
            "Modelo": nombre_modelo,
            "MAE_g_t": mean_absolute_error(y, oof),
            "RMSE_g_t": mean_squared_error(y, oof) ** 0.5,
            "MedAE_g_t": median_absolute_error(y, oof),
            "R2": r2_score(y, oof),
            "RMSE_log": mean_squared_error(y_log, np.log1p(oof)) ** 0.5,
        })
    return pd.DataFrame(filas), predicciones

kfold = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)
div_random = list(kfold.split(X))
groupfold = GroupKFold(n_splits=candidatos["Bloque_espacial"].nunique())
div_espacial = list(groupfold.split(X, y_log, groups=grupos))

metricas_random, pred_random = evaluar_esquema("Aleatoria", div_random)
metricas_espacial, pred_espacial = evaluar_esquema("Espacial", div_espacial)
comparacion = pd.concat([metricas_random, metricas_espacial], ignore_index=True)
display(comparacion.round(3).sort_values(["Validacion", "MAE_g_t"]))


In [ ]:
espacial_aprendidos = metricas_espacial.loc[metricas_espacial["Modelo"].ne("Base mediana")]
mejor_nombre = espacial_aprendidos.sort_values("MAE_g_t").iloc[0]["Modelo"]
mae_mejor = float(espacial_aprendidos.set_index("Modelo").loc[mejor_nombre, "MAE_g_t"])
mae_base = float(metricas_espacial.set_index("Modelo").loc["Base mediana", "MAE_g_t"])
mejora_pct = 100 * (mae_base - mae_mejor) / mae_base
supera_base = mae_mejor < mae_base
mejora_material = mejora_pct >= 5.0

print("Mejor algoritmo aprendido según MAE espacial:", mejor_nombre)
print(f"MAE espacial del algoritmo: {mae_mejor:.3f} g/t")
print(f"MAE espacial de la base: {mae_base:.3f} g/t")
print(f"Mejora frente a la base: {mejora_pct:.1f}%")
if mejora_material:
    print("Resultado: el algoritmo logra una mejora material frente a la base, pero todavía requiere validación geológica externa.")
else:
    print("Resultado: la mejora es marginal o inexistente. No debe presentarse como modelo predictivo listo para uso.")


## 6. Importancia de variables con pruebas espaciales

La importancia se calcula permutando cada variable únicamente en los bloques de prueba. El resultado indica cuánto aumenta el error logarítmico cuando se rompe la información de una variable.


In [ ]:
importancias = []
for numero_fold, (entrenamiento, prueba) in enumerate(div_espacial, start=1):
    estimador = clone(pipelines[mejor_nombre])
    estimador.fit(X.iloc[entrenamiento], y_log[entrenamiento])
    resultado = permutation_importance(
        estimador, X.iloc[prueba], y_log[prueba],
        scoring="neg_mean_absolute_error", n_repeats=12,
        random_state=SEMILLA + numero_fold, n_jobs=-1,
    )
    for variable, media, sd in zip(variables_modelo, resultado.importances_mean, resultado.importances_std):
        importancias.append({"Fold": numero_fold, "Variable": variable, "Importancia": media, "SD": sd})

importancia_resumen = (
    pd.DataFrame(importancias).groupby("Variable")
    .agg(Importancia_media=("Importancia", "mean"), Importancia_sd=("Importancia", "std"))
    .reset_index().sort_values("Importancia_media", ascending=False)
)
display(importancia_resumen.round(4))


## 7. Entrenamiento final y artefactos internos

El mejor algoritmo aprendido se ajusta con todos los candidatos. Este archivo es **interno**: el origen de coordenadas necesario para usar nuevas muestras no debe publicarse.


In [ ]:
RESULTADOS = Path("resultados_fase3_alpacay")
FIGURAS = RESULTADOS / "figuras"
MODELOS = RESULTADOS / "modelo_interno"
FIGURAS.mkdir(parents=True, exist_ok=True)
MODELOS.mkdir(parents=True, exist_ok=True)

modelo_final = clone(pipelines[mejor_nombre]).fit(X, y_log)
joblib.dump(modelo_final, MODELOS / "modelo_au_alpacay.joblib")

configuracion = {
    "zona_analitica": "ALPACAY",
    "modelo": mejor_nombre,
    "objetivo_entrenamiento": "log1p(Au_g_t)",
    "variables": variables_modelo,
    "origen_coordenadas_utm_interno": origen,
    "pca_componentes": pca.components_.tolist(),
    "pca_media_utm_interna": pca.mean_.tolist(),
    "tolerancia_tramo_m": 20.0,
    "bloques_validacion": int(candidatos["Bloque_espacial"].nunique()),
    "mae_espacial_modelo_g_t": mae_mejor,
    "mae_espacial_base_g_t": mae_base,
    "supera_modelo_base": bool(supera_base),
    "mejora_material_minima_5pct": bool(mejora_material),
    "advertencia": "Uso experimental interno; no constituye estimación de recursos o reservas.",
}
(MODELOS / "config_modelo_INTERNO.json").write_text(json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")

comparacion.to_csv(RESULTADOS / "metricas_validacion.csv", index=False)
importancia_resumen.to_csv(RESULTADOS / "importancia_variables_espacial.csv", index=False)
resumen_bloques.to_csv(RESULTADOS / "resumen_bloques_espaciales.csv", index=False)


## 8. Visualizaciones del modelo

Las salidas utilizan coordenadas relativas y códigos anónimos.


In [ ]:
figuras = []
def guardar_figura(nombre):
    ruta = FIGURAS / nombre
    plt.tight_layout()
    plt.savefig(ruta, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()
    figuras.append(ruta)

# Figura 1: bloques espaciales
plt.figure(figsize=(12, 5.5))
sc = plt.scatter(candidatos["Este_rel_m"], candidatos["Norte_rel_m"], c=candidatos["Bloque_espacial"], cmap="tab10", s=38)
plt.colorbar(sc, label="Bloque espacial")
plt.xlabel("Este relativo (m)")
plt.ylabel("Norte relativo (m)")
plt.title("Bloques usados para validación espacial – ALPACAY")
guardar_figura("01_bloques_validacion_espacial.png")


In [ ]:
# Figura 2: optimismo de la validación aleatoria
pivote = comparacion.pivot(index="Modelo", columns="Validacion", values="MAE_g_t").loc[list(modelos)]
ax = pivote.plot(kind="bar", figsize=(11, 5.5), color=[AZUL, ORO])
ax.set_ylabel("MAE (g/t), menor es mejor")
ax.set_xlabel("Modelo")
ax.set_title("Validación aleatoria frente a validación espacial")
plt.xticks(rotation=25, ha="right")
guardar_figura("02_aleatoria_vs_espacial.png")


In [ ]:
# Figura 3: métricas espaciales por modelo
orden_metricas = metricas_espacial.sort_values("MAE_g_t")
plt.figure(figsize=(10, 5.5))
colores = [VERDE if m == mejor_nombre else ("#9E9E9E" if m == "Base mediana" else AZUL) for m in orden_metricas["Modelo"]]
plt.bar(orden_metricas["Modelo"], orden_metricas["MAE_g_t"], color=colores)
plt.ylabel("MAE espacial (g/t)")
plt.title("Comparación de modelos con bloques espaciales")
plt.xticks(rotation=25, ha="right")
guardar_figura("03_comparacion_modelos_espacial.png")


In [ ]:
# Figura 4: observado frente a predicción OOF espacial
pred_mejor = pred_espacial[mejor_nombre]
limite = max(y.max(), pred_mejor.max())
plt.figure(figsize=(7, 7))
plt.scatter(y, pred_mejor, c=grupos, cmap="tab10", alpha=0.72, s=38, edgecolors="white", linewidths=0.3)
plt.plot([0.03, limite], [0.03, limite], "--", color=ROJO, label="Predicción perfecta")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Au observado (g/t)")
plt.ylabel("Au predicho fuera de muestra (g/t)")
plt.title(f"Predicción espacial fuera de muestra – {mejor_nombre}")
plt.legend()
guardar_figura("04_observado_vs_predicho.png")


In [ ]:
# Figura 5: residuos espaciales
residuo = y - pred_mejor
lim = np.quantile(np.abs(residuo), 0.95)
plt.figure(figsize=(12, 5.5))
sc = plt.scatter(
    candidatos["Este_rel_m"], candidatos["Norte_rel_m"], c=np.clip(residuo, -lim, lim),
    cmap="coolwarm", vmin=-lim, vmax=lim, s=42, edgecolors="white", linewidths=0.25,
)
plt.colorbar(sc, label="Residuo observado - predicho (g/t), recortado P95")
plt.xlabel("Este relativo (m)")
plt.ylabel("Norte relativo (m)")
plt.title("Distribución espacial de residuos fuera de muestra")
guardar_figura("05_residuos_espaciales.png")


In [ ]:
# Figura 6: importancia espacial de variables
imp_plot = importancia_resumen.sort_values("Importancia_media").tail(12)
plt.figure(figsize=(9, 6))
plt.barh(imp_plot["Variable"], imp_plot["Importancia_media"], color=AZUL)
plt.axvline(0, color="#666666", linewidth=1)
plt.xlabel("Aumento del MAE logarítmico al permutar")
plt.title(f"Importancia de variables en pruebas espaciales – {mejor_nombre}")
guardar_figura("06_importancia_variables.png")


## 9. Predicciones anónimas y reporte

Las predicciones son fuera de muestra: cada valor se obtuvo cuando su bloque espacial no participaba en el entrenamiento.


In [ ]:
predicciones_salida = pd.DataFrame({
    "MuestraAnonima": [f"M-{i+1:04d}" for i in range(len(candidatos))],
    "ZonaAnalitica": "ALPACAY",
    "Labor": candidatos["Labor"],
    "Nivel": candidatos["Nivel"],
    "BloqueEspacial": candidatos["Bloque_espacial"],
    "EsteRel_m": candidatos["Este_rel_m"],
    "NorteRel_m": candidatos["Norte_rel_m"],
    "CotaRel_m": candidatos["Cota_rel_m"],
    "AuObservado_g_t": y,
    "AuPredichoOOF_g_t": pred_mejor,
    "Residuo_g_t": y - pred_mejor,
})
predicciones_salida.to_csv(RESULTADOS / "predicciones_espaciales_anonimizadas.csv", index=False)
display(predicciones_salida.head().round(3))


In [ ]:
def imagen_embebida(ruta):
    codificada = base64.b64encode(Path(ruta).read_bytes()).decode("ascii")
    return f'<section><img src="data:image/png;base64,{codificada}" alt="{Path(ruta).stem}"></section>'

tabla_metricas_html = comparacion.round(3).sort_values(["Validacion", "MAE_g_t"]).to_html(index=False)
tabla_importancia_html = importancia_resumen.head(12).round(4).to_html(index=False)
imagenes_html = "".join(imagen_embebida(ruta) for ruta in figuras)
estado = (
    f"El algoritmo {mejor_nombre} logra una mejora material de {mejora_pct:.1f}% frente al MAE espacial de la base."
    if mejora_material else
    f"El algoritmo {mejor_nombre} solo cambia el MAE espacial en {mejora_pct:.1f}% frente a la base; el resultado es marginal y no está listo para uso predictivo."
)
color_estado = "#E2F0D9" if mejora_material else "#FCE4D6"

reporte = f'''<!doctype html><html lang="es"><head><meta charset="utf-8">
<title>Fase 3 – Machine Learning ALPACAY</title><style>
body{{font-family:Arial,sans-serif;max-width:1100px;margin:40px auto;color:#1f2937;line-height:1.45}}
h1,h2{{color:{NAVY}}}.estado{{background:{color_estado};border-left:5px solid {VERDE if mejora_material else ROJO};padding:12px}}
.nota{{background:#fff2cc;border-left:5px solid {ORO};padding:12px}}
table{{border-collapse:collapse;width:100%;margin:18px 0}}th{{background:{NAVY};color:white}}
td,th{{padding:8px;border-bottom:1px solid #d9e2f3;text-align:left}}
img{{width:100%;max-width:1000px;border:1px solid #e5e7eb;margin:12px 0}}
footer{{margin-top:30px;color:#6b7280;font-size:.9rem}}</style></head><body>
<h1>Proyecto Minería 4.0 – Fase 3</h1>
<p>Machine Learning espacial para estimación experimental de Au en la única zona analítica <strong>ALPACAY</strong>.</p>
<div class="estado"><strong>Resultado de validación:</strong> {estado}</div>
<div class="nota"><strong>Criterio:</strong> la validación espacial es la referencia para seleccionar modelos. Las leyes altas se conservaron y se aplicó log1p al objetivo.</div>
<h2>Métricas</h2>{tabla_metricas_html}
<h2>Importancia espacial</h2>{tabla_importancia_html}
<h2>Visualizaciones</h2>{imagenes_html}
<footer>Resultados experimentales. No constituyen estimación de recursos o reservas ni sustituyen la interpretación geológica. Coordenadas mostradas de forma relativa.</footer>
</body></html>'''

RUTA_REPORTE = RESULTADOS / "REPORTE_FASE3_MACHINE_LEARNING_ALPACAY.html"
RUTA_REPORTE.write_text(reporte, encoding="utf-8")
print("Reporte generado:", RUTA_REPORTE)
display(HTML(f"<p><strong>{estado}</strong></p>"))


## 10. Paquete final

El ZIP incluye métricas, predicciones anónimas, gráficos, reporte y el modelo interno. No publiques la carpeta `modelo_interno`, porque contiene parámetros ligados al sistema de coordenadas real.


In [ ]:
RUTA_ZIP = Path("FASE3_ALPACAY_MACHINE_LEARNING_RESULTADOS.zip")
with zipfile.ZipFile(RUTA_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for archivo in RESULTADOS.rglob("*"):
        if archivo.is_file():
            zf.write(archivo, archivo.relative_to(RESULTADOS.parent))

print("Paquete generado:", RUTA_ZIP)
if EN_COLAB:
    files.download(str(RUTA_ZIP))


## Criterio para la siguiente etapa

El modelo solo debe avanzar a uso piloto si mejora al modelo base con validación espacial y si un geólogo/minero valida que las variables y los bloques representan correctamente la continuidad de la veta. La incorporación del plano DXF y variables geológicas adicionales puede mejorar la siguiente iteración.
